# Router Bias Intervention — Phase 2 of Paper 2

**Goal**: amplify top-helper experts + suppress top-anti experts at L18-L20 (where discriminative power peaks per Phase 1). Measure accuracy Δ vs baseline.

**Approach**: forward hook on each MoE router. Intercept `(logits, weights, indices)` output, add bias vector to logits, recompute top-k selection with new bias, return modified tuple.

**Expected Pareto gain (Phase 1 analysis)**: +2 to +5pp. Top experts have ~10pp activation differential — amplifying 5-10 simultaneously should shift some wrong→correct.

**Budget**: ~2h compute (sweep of 6 bias magnitudes × N=100).

**Prerequisite**: Phase 1 notebook ran successfully → `expert_discriminative.npz` exists in `/content/expert_profiling/`.

## Cell 1 — Setup + load discriminative data

In [ ]:
# Assumes Phase 1 session: model, tok, rollouts, moe_layers, N_EXPERTS, TOP_K, handles already defined
# If fresh session, rerun Phase 1 cells 1-4 first

import numpy as np
from pathlib import Path
import json, time, random
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

OUT = Path('/content/router_bias'); OUT.mkdir(exist_ok=True)
PHASE1_DIR = Path('/content/expert_profiling')

# Load Phase 1 discriminative data
data = np.load(PHASE1_DIR / 'expert_freq_lasttoken.npz', allow_pickle=True)
expert_freq_last = data['expert_freq']
rollout_correct = data['correct'].astype(bool)
moe_layers_phase1 = list(data['moe_layers'])

mean_correct = expert_freq_last[rollout_correct].mean(axis=0)
mean_wrong = expert_freq_last[~rollout_correct].mean(axis=0)
discriminative = mean_correct - mean_wrong  # [n_layers, n_experts]
print(f'discriminative shape: {discriminative.shape}')
print(f'range: {discriminative.min():.4f} to {discriminative.max():.4f}')

## Cell 2 — Build bias vectors per configuration

For each (target layer range, top-K helpers, top-K antis, bias magnitude) config, create a `[n_moe_layers, n_experts]` bias matrix.

In [ ]:
# Target: reasoning + commitment zone (L18-L20 shows strongest effects)
# Helper experts: top-10 per layer with highest discriminative
# Anti experts: top-10 per layer with most negative discriminative

def build_bias(target_layers, n_helpers, n_antis, helper_bias, anti_bias):
    """Build [n_moe_layers, n_experts] bias matrix."""
    bias = np.zeros((len(moe_layers_phase1), N_EXPERTS), dtype=np.float32)
    for L in target_layers:
        if L not in moe_layers_phase1: continue
        li = moe_layers_phase1.index(L)
        disc = discriminative[li]
        helpers = np.argsort(-disc)[:n_helpers]
        antis = np.argsort(disc)[:n_antis]
        bias[li, helpers] = helper_bias
        bias[li, antis] = anti_bias
    return torch.from_numpy(bias).cuda().to(torch.bfloat16)

# Configurations to sweep
CONFIGS = [
    ('baseline', None, None, None, None, 0.0, 0.0),
    # Mild bias on L18-20 commitment zone
    ('L18-20_h10_bias1', [18,19,20], 10, 10, 1.0, -1.0, None),
    ('L18-20_h10_bias2', [18,19,20], 10, 10, 2.0, -2.0, None),
    ('L18-20_h10_bias5', [18,19,20], 10, 10, 5.0, -5.0, None),
    # Broader: L15-L20
    ('L15-20_h10_bias2', [15,16,17,18,19,20], 10, 10, 2.0, -2.0, None),
    # Helpers only (no anti suppression)
    ('L18-20_h10_bias2_helpersOnly', [18,19,20], 10, 0, 2.0, 0.0, None),
    # Antis only (no helper amplification)
    ('L18-20_a10_bias2_antisOnly', [18,19,20], 0, 10, 0.0, -2.0, None),
]

config_biases = {}
for name, layers, nh, na, hb, ab, _ in CONFIGS:
    if name == 'baseline':
        config_biases[name] = None
    else:
        config_biases[name] = build_bias(layers, nh, na, hb, ab)

print(f'{len(CONFIGS)} configs built')
# Inspect L18 of bias2 config
test_bias = config_biases['L18-20_h10_bias2'].cpu().numpy()
L18_idx = moe_layers_phase1.index(18)
print(f'\nL18 bias (L18-20_h10_bias2): {(test_bias[L18_idx]!=0).sum()} non-zero')
print(f'  helpers at L18: {np.where(test_bias[L18_idx]>0)[0]}')
print(f'  antis at L18: {np.where(test_bias[L18_idx]<0)[0]}')

## Cell 3 — Install router bias hooks (modifies router output)

In [ ]:
# Remove any previous hooks (Phase 1 profiling pre-hooks)
if 'handles' in dir():
    for h in handles:
        try: h.remove()
        except: pass
    handles = []

# New bias hook: forward hook on router (not pre), modifies output tuple
class RouterBiasController:
    """Hook on each MoE router. Apply bias to logits, recompute top-k."""
    def __init__(self, layer_idx, layer_idx_in_bias_matrix):
        self.layer_idx = layer_idx
        self.li = layer_idx_in_bias_matrix
        self.bias_matrix = None  # [n_layers, n_experts] — slice [li] used
        self.active = False
    def set(self, bias_matrix):
        self.bias_matrix = bias_matrix
        self.active = True if bias_matrix is not None else False
    def off(self):
        self.active = False
    def make_hook(self):
        def hook(module, inp, out):
            if not self.active or self.bias_matrix is None: return out
            if not isinstance(out, tuple) or len(out) < 3: return out
            logits, _, _ = out  # (logits [T, n_experts], weights [T,8], indices [T,8])
            # Apply bias on last token only
            biased_logits = logits.clone()
            bias_vec = self.bias_matrix[self.li].to(biased_logits.dtype)
            biased_logits[-1, :] = biased_logits[-1, :] + bias_vec
            # Recompute top-k
            top_k_vals, top_k_idx = biased_logits.topk(module.top_k, dim=-1)
            top_k_weights = F.softmax(top_k_vals.float(), dim=-1).to(logits.dtype)
            return (biased_logits, top_k_weights, top_k_idx)
        return hook

controllers = {}
handles = []
for li, L in enumerate(moe_layers_phase1):
    ctrl = RouterBiasController(L, li)
    controllers[L] = ctrl
    h = all_layers[L].mlp.gate.register_forward_hook(ctrl.make_hook())
    handles.append(h)

def apply_config(bias_matrix):
    for L, ctrl in controllers.items():
        ctrl.set(bias_matrix)

def off_all():
    for ctrl in controllers.values():
        ctrl.off()

print(f'✅ {len(handles)} router bias hooks installed')

## Cell 4 — Eval loop: sweep configs on Stage B 10-opt

In [ ]:
# Filter rollouts to 10-option questions where Stage B had ≥1 correct (reasonable baseline)
eval_questions = [r for r in rollouts if len(r['options']) == 10]
random.seed(42)
random.shuffle(eval_questions)
# Unique questions (dedupe)
seen_q = set(); unique_qs = []
for r in eval_questions:
    if r['question'] not in seen_q:
        seen_q.add(r['question'])
        unique_qs.append(r)
eval_sample = unique_qs[:100]
print(f'{len(eval_sample)} unique questions for eval')

letter_ids_map = {L_: tok(L_, add_special_tokens=False).input_ids[0] for L_ in 'ABCDEFGHIJ'}

results = []
t0 = time.time()
for i, q in enumerate(tqdm(eval_sample, desc='router-bias eval')):
    try:
        prompt = ("Answer the following multiple-choice question. "
            "Give ONLY the letter of the correct answer.\n\n"
            f"Question: {q['question']}\n\nOptions:\n"
            + '\n'.join(f'{chr(65+j)}. {opt}' for j, opt in enumerate(q['options']))
            + "\n\nAnswer: \\boxed{")
        p = tok.apply_chat_template([{'role':'user','content':prompt}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        row = dict(idx=i, gold=q['gold_letter'])
        for cfg_name, *_ in CONFIGS:
            apply_config(config_biases[cfg_name])
            with torch.no_grad():
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]
            letter_logits = {L_: logits[letter_ids_map[L_]].item() for L_ in 'ABCDEFGHIJ'[:10]}
            pred = max(letter_logits, key=letter_logits.get)
            row[f'{cfg_name}_pred'] = pred
            row[f'{cfg_name}_correct'] = (pred == q['gold_letter'])
        off_all()
        results.append(row)
        if (i+1) % 10 == 0:
            accs = {n: sum(r[f'{n}_correct'] for r in results)/len(results) for n,*_ in CONFIGS}
            top3 = sorted(accs.items(), key=lambda x: -x[1])[:3]
            print(f'  [{i+1}/{len(eval_sample)}] top3: ' + ' | '.join(f'{n}={a*100:.0f}%' for n,a in top3))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception as e:
        print(f'item {i}: {e}'); continue

with open(OUT/'bias_results.json', 'w') as f:
    json.dump(dict(n=len(results), configs=[c[0] for c in CONFIGS],
                   results=results, wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ Done in {(time.time()-t0)/60:.1f} min')

## Cell 5 — Analysis + verdict

In [ ]:
from scipy.stats import binomtest

print(f'=== Router Bias Intervention (N={len(results)}) ===\n')
bc = [r['baseline_correct'] for r in results]
base_acc = sum(bc) / len(results) * 100
print(f'baseline              : {base_acc:.1f}%\n')
print(f'{"config":35s} {"acc%":>6s}  {"Δpp":>6s}  {"gain":>5s}  {"lost":>5s}  {"p":>7s}')
stats = []
for (n,*_) in CONFIGS:
    if n == 'baseline': continue
    cc = [r[f'{n}_correct'] for r in results]
    acc = sum(cc)/len(results)
    delta = (acc - base_acc/100)*100
    g = sum(1 for b,c in zip(bc,cc) if not b and c)
    l = sum(1 for b,c in zip(bc,cc) if b and not c)
    p = binomtest(g, g+l, p=0.5, alternative='two-sided').pvalue if g+l > 0 else 1.0
    stats.append((n, acc*100, delta, g, l, p))
stats.sort(key=lambda r: -r[2])
for n, acc, d, g, l, p in stats:
    star = ' ***' if d>3 and l==0 else (' **' if d>1.5 and l==0 else (' *' if d>0.5 and l==0 else ''))
    print(f'{n:35s} {acc:5.1f}%  {d:+5.1f}  {g:>5d}  {l:>5d}  {p:>7.3f}{star}')

best_pareto = max([s for s in stats if s[4] == 0], key=lambda r: r[2], default=None)
best_overall = max(stats, key=lambda r: r[2])
print('\n=== Verdict ===')
if best_pareto and best_pareto[2] > 3:
    print(f'*** BREAKTHROUGH: {best_pareto[0]} → {best_pareto[2]:+.1f}pp Pareto')
    print('    Router bias intervention amplifies beyond +2.7pp amnesic ceiling.')
elif best_pareto and best_pareto[2] > 1.5:
    print(f'**  MODERATE: {best_pareto[0]} → {best_pareto[2]:+.1f}pp Pareto')
elif best_pareto and best_pareto[2] > 0:
    print(f'*   Small positive: {best_pareto[0]} → {best_pareto[2]:+.1f}pp Pareto')
else:
    print(f'--  No Pareto positive: best overall {best_overall[0]} {best_overall[2]:+.1f}pp (lost {best_overall[4]})')
    print('    Fixed-bias router intervention does not work on aggregate reasoning.')